## utils/img_read.py

###  img_read() 测试

In [5]:
import os
import shutil
import numpy as np
from PIL import Image
from SFDFusion.utils.img_read import img_read as img_read_torch

def generate_torch_baseline():
    """
    在 PyTorch 环境下运行，生成基准测试文件。
    """
    print("--- PyTorch 环境：开始生成基准数据 ---")
    
    # 1. 创建临时测试图片
    temp_dir = 'temp_test_data'
    if not os.path.exists(temp_dir):
        os.makedirs(temp_dir)
    
    image_path = os.path.join(temp_dir, 'test_image.png')
    if not os.path.exists(image_path):
        print(f"创建测试图片于: {image_path}")
        dummy_array = np.random.randint(0, 256, (20, 20, 3), dtype=np.uint8)
        Image.fromarray(dummy_array, 'RGB').save(image_path)

    # 2. 使用 PyTorch 代码读取图片
    print("读取 RGB 模式...")
    torch_rgb = img_read_torch(image_path, 'RGB')
    
    print("读取 L (灰度) 模式...")
    torch_l = img_read_torch(image_path, 'L')
    
    print("读取 YCbCr 模式...")
    torch_y, torch_cbcr = img_read_torch(image_path, 'YCbCr')

    # 3. 将所有结果保存到 .npz 文件
    output_path = 'pyTest/img_read_results.npz'
    np.savez(
        output_path,
        rgb=torch_rgb.numpy(),
        l=torch_l.numpy(),
        y=torch_y.numpy(),
        cbcr=torch_cbcr.numpy()
    )
    print(f"✅ PyTorch 基准数据已保存至: {output_path}")
    print("--- PyTorch 环境：任务完成 ---")


if __name__ == '__main__':
    # 请在您的 PyTorch 环境中运行此脚本
    generate_torch_baseline()

--- PyTorch 环境：开始生成基准数据 ---
读取 RGB 模式...
读取 L (灰度) 模式...
读取 YCbCr 模式...
✅ PyTorch 基准数据已保存至: pyTest/img_read_results.npz
--- PyTorch 环境：任务完成 ---


### img_save() 测试

In [6]:
import os
import numpy as np
from PIL import Image
# 假设您的函数保存在 SFDFusion/utils/img_read.py
from SFDFusion.utils.img_read import img_read as img_read_pytorch, img_save as img_save_pytorch

def generate_pytorch_saved_image():
    print("--- PyTorch 环境：开始生成待测数据 (img_save) ---")
    
    test_dir = 'pyTest/save_test'
    if not os.path.exists(test_dir):
        os.makedirs(test_dir)
        
    original_image_path = os.path.join(test_dir, 'original_for_save_test.png')
    if not os.path.exists(original_image_path):
        print(f"创建原始测试图片: {original_image_path}")
        dummy_array = np.random.randint(0, 256, (100, 80, 3), dtype=np.uint8)
        Image.fromarray(dummy_array, 'RGB').save(original_image_path)
    else:
        print(f"使用已存在的原始测试图片: {original_image_path}")

    print("使用 PyTorch 读取原始图片...")
    torch_tensor = img_read_pytorch(original_image_path, 'RGB')
    
    # ======================================================================
    # ========================= 关键修正部分 ===============================
    # ======================================================================
    # 模拟您主项目中的做法：在调用 img_save 之前，手动将 Tensor 转换为 NumPy 数组
    print("正在将 PyTorch Tensor 转换为 NumPy 数组...")
    
    # 1. 将 Tensor 移动到 CPU 并转换为 NumPy 数组
    #    Tensor 格式: [C, H, W], 值范围 [0.0, 1.0]
    numpy_array = torch_tensor.cpu().numpy()
    
    # 2. 转换维度以匹配 PIL 的要求
    #    从 [C, H, W] 转换为 [H, W, C]
    numpy_array = numpy_array.transpose(1, 2, 0)
    
    # 3. 将值范围从 [0.0, 1.0] 转换回 [0, 255]
    numpy_array = (numpy_array * 255).astype(np.uint8)
    
    print("使用 PyTorch 保存图片...")
    # 现在传递的是 NumPy 数组，与您主项目的 img_save 函数完全兼容
    img_save_pytorch(numpy_array, 'torch_saved.png', test_dir, mode='RGB')
    # ======================================================================
    
    print(f"✅ PyTorch 输出图片已保存至: {os.path.join(test_dir, 'torch_saved.png')}")
    print("--- PyTorch 环境：任务完成 ---")

if __name__ == '__main__':
    generate_pytorch_saved_image()

--- PyTorch 环境：开始生成待测数据 (img_save) ---
使用已存在的原始测试图片: pyTest/save_test/original_for_save_test.png
使用 PyTorch 读取原始图片...
正在将 PyTorch Tensor 转换为 NumPy 数组...
使用 PyTorch 保存图片...
✅ PyTorch 输出图片已保存至: pyTest/save_test/torch_saved.png
--- PyTorch 环境：任务完成 ---


## utils/evaluator.py

In [10]:
import os
import numpy as np
import json
from SFDFusion.utils.evaluator import Evaluator

def generate_base_metrics():
    """
    使用原始的 NumPy/SciPy/Sklearn 实现来生成基准评估指标。
    """
    print("--- NumPy 环境：开始生成基准评估指标 ---")

    # 1. 创建统一的测试数据
    # 使用固定的随机种子确保每次生成的图像都一样
    np.random.seed(42) 
    # 创建三张 64x64 的灰度图，数值范围在 0-255 之间
    img_a = np.random.rand(64, 64) * 255
    img_b = np.random.rand(64, 64) * 255
    # "融合后"的图像 F 可以是 A 和 B 的简单平均
    img_f = (img_a + img_b) / 2
    
    # 确保数据类型是 float64，以获得最高精度
    img_a, img_b, img_f = img_a.astype(np.float64), img_b.astype(np.float64), img_f.astype(np.float64)

    print("已创建三张 64x64 的测试图像 (img_a, img_b, img_f)。")

    # 2. 计算所有指标
    print("正在计算所有评估指标...")
    results = {}

    # 单图像指标
    results['EN'] = Evaluator.EN(img_f)
    results['SD'] = Evaluator.SD(img_f)
    results['SF'] = Evaluator.SF(img_f)
    results['AG'] = Evaluator.AG(img_f)

    # 多图像指标
    results['MI'] = Evaluator.MI(img_f, img_a, img_b)
    results['MSE'] = Evaluator.MSE(img_f, img_a, img_b)
    results['CC'] = Evaluator.CC(img_f, img_a, img_b)
    results['PSNR'] = Evaluator.PSNR(img_f, img_a, img_b)
    results['SCD'] = Evaluator.SCD(img_f, img_a, img_b)
    results['VIFF'] = Evaluator.VIFF(img_f, img_a, img_b)
    results['Qabf'] = Evaluator.Qabf(img_f, img_a, img_b)
    results['SSIM'] = Evaluator.SSIM(img_f, img_a, img_b)
    
    print("所有指标计算完成。")
    
    # 3. 保存结果到文件
    output_dir = 'pyTest/eval_test'
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        
    # 保存测试图像，以便 Jittor 脚本可以加载完全相同的数据
    np.save(os.path.join(output_dir, 'img_a.npy'), img_a)
    np.save(os.path.join(output_dir, 'img_b.npy'), img_b)
    np.save(os.path.join(output_dir, 'img_f.npy'), img_f)
    print(f"测试图像已保存至: {output_dir}")

    # 保存计算出的指标
    results_path = os.path.join(output_dir, 'numpy_results.json')
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=4)

    print(f"✅ 基准指标已保存至: {results_path}")
    print("--- NumPy 环境：任务完成 ---")


if __name__ == '__main__':
    generate_base_metrics()

--- NumPy 环境：开始生成基准评估指标 ---
已创建三张 64x64 的测试图像 (img_a, img_b, img_f)。
正在计算所有评估指标...
所有指标计算完成。
测试图像已保存至: pyTest/eval_test
✅ 基准指标已保存至: pyTest/eval_test/numpy_results.json
--- NumPy 环境：任务完成 ---


/home/wyx/miniconda3/envs/SFD/lib/python3.10/site-packages/sklearn/metrics/cluster/_supervised.py:66: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and continuous values for target
  warnings.warn(msg, UserWarning)
/home/wyx/miniconda3/envs/SFD/lib/python3.10/site-packages/sklearn/metrics/cluster/_supervised.py:66: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and continuous values for target
  warnings.warn(msg, UserWarning)


## utils/loss.py

In [5]:
import torch
import torch.nn.functional as F
import numpy as np
from SFDFusion.utils.loss import Sobelxy

def run_pytorch_debug():
    """
    分解 PyTorch PixelGradLoss 的计算，打印每一步的中间结果。
    """
    print("\n--- PyTorch 调试环境 ---")
    
    # 确保使用 CUDA
    if not torch.cuda.is_available():
        print("❌ 错误: PyTorch 未找到 CUDA。")
        return
    device = torch.device("cuda")
    print("PyTorch 使用 CUDA")

    # 1. 创建与之前完全相同的输入数据
    np.random.seed(42)
    image_vis_np = np.random.rand(2, 1, 64, 64).astype('float32')
    image_ir_np = np.random.rand(2, 1, 64, 64).astype('float32')
    fus_img_np = np.random.rand(2, 1, 64, 64).astype('float32')
    
    image_vis = torch.from_numpy(image_vis_np).to(device)
    image_ir = torch.from_numpy(image_ir_np).to(device)
    fus_img = torch.from_numpy(fus_img_np).to(device)

    # 2. 分解 PixelGradLoss 计算过程
    sobel_fn = Sobelxy().to(device)
    
    print("\n--- 开始分解计算 ---")
    y = image_vis[:, :1]
    print(f"[Pytorch] y.sum(): {y.sum().item():.8f}")
    
    x_in = torch.max(y, image_ir)
    print(f"[Pytorch] x_in.sum(): {x_in.sum().item():.8f}")
    
    # 手动计算 loss_in
    loss_in_abs_sum = torch.abs(x_in - fus_img).sum()
    loss_in_numel = x_in.numel()
    loss_in = loss_in_abs_sum / loss_in_numel
    print(f"[Pytorch] loss_in_abs_sum: {loss_in_abs_sum.item():.8f}")
    print(f"[Pytorch] loss_in_numel: {loss_in_numel}")
    print(f"[Pytorch] loss_in: {loss_in.item():.8f}")
    
    gy = sobel_fn(y)
    print(f"[Pytorch] gy.sum(): {gy.sum().item():.8f}")
    
    gir = sobel_fn(image_ir)
    print(f"[Pytorch] gir.sum(): {gir.sum().item():.8f}")

    gf = sobel_fn(fus_img)
    print(f"[Pytorch] gf.sum(): {gf.sum().item():.8f}")
    
    gtarget = torch.max(gy, gir)
    print(f"[Pytorch] gtarget.sum(): {gtarget.sum().item():.8f}")
    
    # 手动计算 loss_grad
    loss_grad_abs_sum = torch.abs(gtarget - gf).sum()
    loss_grad_numel = gtarget.numel()
    loss_grad = loss_grad_abs_sum / loss_grad_numel
    print(f"[Pytorch] loss_grad_abs_sum: {loss_grad_abs_sum.item():.8f}")
    print(f"[Pytorch] loss_grad_numel: {loss_grad_numel}")
    print(f"[Pytorch] loss_grad: {loss_grad.item():.8f}")
    
    final_loss = 5 * loss_in + 10 * loss_grad
    print(f"\n[Pytorch] 最终 PixelGradLoss: {final_loss.item():.8f}")
    print("--- PyTorch 调试结束 ---")


if __name__ == '__main__':
    run_pytorch_debug()


--- PyTorch 调试环境 ---
PyTorch 使用 CUDA

--- 开始分解计算 ---
[Pytorch] y.sum(): 4053.58325195
[Pytorch] x_in.sum(): 5432.59375000
[Pytorch] loss_in_abs_sum: 2682.51269531
[Pytorch] loss_in_numel: 8192
[Pytorch] loss_in: 0.32745516
[Pytorch] gy.sum(): 13803.26562500
[Pytorch] gir.sum(): 13735.91406250
[Pytorch] gf.sum(): 13718.44726562
[Pytorch] gtarget.sum(): 17614.49023438
[Pytorch] loss_grad_abs_sum: 7939.55126953
[Pytorch] loss_grad_numel: 8192
[Pytorch] loss_grad: 0.96918350

[Pytorch] 最终 PixelGradLoss: 11.32911110
--- PyTorch 调试结束 ---


### 高斯窗口检查

In [7]:
import torch
import torch.nn.functional as F

def create_gaussian_window_pytorch(window_size, channel, sigma):
    """
    一个标准的、在 PyTorch 中生成高斯窗口的实现。
    """
    # 创建一维高斯核
    gauss_1d = torch.exp(-(torch.arange(window_size) - window_size // 2)**2 / float(2 * sigma**2))
    gauss_1d = gauss_1d / gauss_1d.sum()
    
    # 将一维核转换为二维
    gauss_2d = gauss_1d.unsqueeze(1).mm(gauss_1d.unsqueeze(0)).float().unsqueeze(0).unsqueeze(0)
    
    # 扩展到多通道
    window = gauss_2d.expand(channel, 1, window_size, window_size).contiguous()
    return window

# --- 参数 (与 SSIMLoss 一致) ---
window_size = 11
sigma = 1.5
channel = 1

pytorch_window = create_gaussian_window_pytorch(window_size, channel, sigma)

print("--- PyTorch Gaussian Window (中心 5x5 值) ---")
# 打印中心区域的值以便比较
print(pytorch_window[0, 0, 3:8, 3:8]) 

--- PyTorch Gaussian Window (中心 5x5 值) ---
tensor([[0.0120, 0.0233, 0.0291, 0.0233, 0.0120],
        [0.0233, 0.0454, 0.0567, 0.0454, 0.0233],
        [0.0291, 0.0567, 0.0708, 0.0567, 0.0291],
        [0.0233, 0.0454, 0.0567, 0.0454, 0.0233],
        [0.0120, 0.0233, 0.0291, 0.0233, 0.0120]])


In [8]:
import torch
import numpy as np
import json
import kornia
from pathlib import Path

# 确保可以从 SFDFusion 目录导入模块
import sys
sys.path.append(str(Path.cwd()))

from SFDFusion.utils.loss import PixelGradLoss, cal_saliency_loss, cal_fre_loss

def get_fft_feature(x):
    """ 使用高级 rfft2 生成合法的频谱 """
    fft_x = torch.fft.rfft2(x, dim=(-2, -1), norm='ortho')
    amp = torch.abs(fft_x)
    pha = torch.angle(fft_x)
    return amp, pha

NUM_TRIALS = 50

def run_pytorch_loss_test():
    if not torch.cuda.is_available():
        print("❌ 错误: PyTorch 未找到 CUDA。")
        return
    device = torch.device("cuda")
    print(f"PyTorch 使用 CUDA，将进行 {NUM_TRIALS} 轮随机数据测试...")

    total_losses = { 'PixelGradLoss': 0.0, 'SaliencyLoss': 0.0, 'FrequencyLoss': 0.0, 'SSIMLoss': 0.0 }
    pixel_grad_loss_fn = PixelGradLoss().to(device)
    ssim_loss_fn = kornia.losses.SSIMLoss(window_size=11).to(device) # 初始化 SSIMLoss

    for i in range(NUM_TRIALS):
        seed = 42 + i
        np.random.seed(seed)
        image_vis_np = np.random.rand(2, 1, 64, 64).astype('float32')
        image_ir_np = np.random.rand(2, 1, 64, 64).astype('float32')
        fus_img_np = np.random.rand(2, 1, 64, 64).astype('float32')
        mask_np = (np.random.rand(2, 1, 64, 64) > 0.5).astype('float32')
        
        image_vis = torch.from_numpy(image_vis_np).to(device)
        image_ir = torch.from_numpy(image_ir_np).to(device)
        fus_img = torch.from_numpy(fus_img_np).to(device)
        mask = torch.from_numpy(mask_np).to(device)
        
        pg_loss = pixel_grad_loss_fn(image_vis, image_ir, fus_img)
        total_losses['PixelGradLoss'] += pg_loss.item()

        sal_loss = cal_saliency_loss(fus_img, image_ir, image_vis, mask)
        total_losses['SaliencyLoss'] += sal_loss.item()

        # --- 正确的测试逻辑 ---
        # 1. 对真实图像进行FFT，得到合法的 amp 和 pha
        amp, pha = get_fft_feature(fus_img)
        # 2. 将合法的 amp 和 pha 送入损失函数
        fre_loss = cal_fre_loss(amp, pha, image_ir, image_vis, mask)
        total_losses['FrequencyLoss'] += fre_loss.item()

        # 计算 SSIM Loss (与训练脚本逻辑对齐，只计算与红外图像的损失)
        ssim_loss = ssim_loss_fn(fus_img, image_ir)
        total_losses['SSIMLoss'] += ssim_loss.item()

    avg_losses = {key: value / NUM_TRIALS for key, value in total_losses.items()}
    
    # 确保输出目录存在
    output_dir = Path('pyTest/loss_test')
    output_dir.mkdir(parents=True, exist_ok=True)
    
    with open(output_dir / 'pytorch_loss_results.json', 'w') as f:
        json.dump(avg_losses, f, indent=4)
    print("\nPyTorch 平均损失结果已保存。")
    print(json.dumps(avg_losses, indent=4))

if __name__ == '__main__':
    run_pytorch_loss_test()

PyTorch 使用 CUDA，将进行 50 轮随机数据测试...

PyTorch 平均损失结果已保存。
{
    "PixelGradLoss": 11.451971893310548,
    "SaliencyLoss": 1.0012910616397859,
    "FrequencyLoss": 1.1976651167869568,
    "SSIMLoss": 0.49784741580486297
}


## utils/saliency.py

暂时不作迁移，设计到u2net

In [1]:
import shutil
from pathlib import Path
import cv2
import numpy as np
from SFDFusion.utils.saliency import Saliency

# 1. 定义测试目录和文件
CWD = Path.cwd()
OUTPUT_DIR = CWD / 'pyTest' / 'saliency_test' / 'output'
SRC_IR_DIR = OUTPUT_DIR / 'src_ir'
DST_PYTORCH_DIR = OUTPUT_DIR / 'dst_pytorch'
IMAGE_NAMES = ['test_image_01.png', 'test_image_02.png']

def setup_environment():
    """准备测试环境：创建目录和随机图片。"""
    print("--- 正在准备 PyTorch 测试环境... ---")
    if OUTPUT_DIR.exists():
        shutil.rmtree(OUTPUT_DIR)
    SRC_IR_DIR.mkdir(parents=True, exist_ok=True)
    DST_PYTORCH_DIR.mkdir(exist_ok=True)

    # 使用固定种子生成可复现的随机图片
    np.random.seed(42)
    img1 = np.random.randint(0, 256, (256, 256), dtype=np.uint8)
    img2 = np.random.randint(0, 256, (300, 200), dtype=np.uint8)
    
    cv2.imwrite(str(SRC_IR_DIR / IMAGE_NAMES[0]), img1)
    cv2.imwrite(str(SRC_IR_DIR / IMAGE_NAMES[1]), img2)
    print(f"  - 环境已在 {OUTPUT_DIR} 创建。")

def run_pytorch_benchmark():
    """运行原始的 PyTorch Saliency 模块。"""
    print("--- 正在运行 PyTorch 基准测试... ---")
    try:
        saliency_detector = Saliency()
        saliency_detector.inference(src=SRC_IR_DIR, dst=DST_PYTORCH_DIR, suffix='png')
        print(f"  - PyTorch 推理完成，结果已保存至: {DST_PYTORCH_DIR}")
    except Exception as e:
        print(f"❌ PyTorch 基准测试失败: {e}")

if __name__ == '__main__':
    setup_environment()
    run_pytorch_benchmark()

--- 正在准备 PyTorch 测试环境... ---
  - 环境已在 /home/wyx/projects/pyTest/saliency_test/output 创建。
--- 正在运行 PyTorch 基准测试... ---


generate mask for test_image_01.png to /home/wyx/projects/pyTest/saliency_test/output/dst_pytorch:   0%|          | 0/2 [00:00<?, ?it/s]/home/wyx/miniconda3/envs/SFD/lib/python3.10/site-packages/torchvision/transforms/functional.py:1603: UserWarning: The default value of the antialias parameter of all the resizing transforms (Resize(), RandomResizedCrop(), etc.) will change from None to True in v0.17, in order to be consistent across the PIL and Tensor backends. To suppress this warning, directly pass antialias=True (recommended, future default), antialias=None (current default, which means False for Tensors and True for PIL), or antialias=False (only works on Tensors - PIL will still use antialiasing). This also applies if you are using the inference transforms from the models weights: update the call to weights.transforms(antialias=True).
  warnings.warn(
/home/wyx/miniconda3/envs/SFD/lib/python3.10/site-packages/torch/nn/functional.py:3769: UserWarning: nn.functional.upsample is dep

  - PyTorch 推理完成，结果已保存至: /home/wyx/projects/pyTest/saliency_test/output/dst_pytorch


## dataset.py

In [9]:
import torch
from torchvision import transforms
from torch.utils.data import Dataset
import numpy as np
from PIL import Image
import os
from pathlib import Path
import logging
import yaml

# 假设原始的 PyTorch Dataset 定义在以下路径
from SFDFusion.dataset import RoadScene as RoadScene_torch
from SFDFusion.configs import from_dict

# --- 配置 ---
# 关键修改 1: 定义要测试的样本数量
NUM_TEST_SAMPLES = 10
# ----------------

CWD = Path.cwd()
OUTPUT_DIR = CWD / 'pyTest' / 'dataset_test' / 'output'
PYTORCH_RESULT_PATH = OUTPUT_DIR / 'pytorch_sample.npz'

logging.basicConfig(level=logging.INFO)

def generate_pytorch_batch():
    """在 PyTorch 环境下运行，生成包含多个样本的待对比数据。"""
    print("--- PyTorch 环境：开始生成批量待测数据 ---")
    
    # 1. 初始化 PyTorch Dataset
    config = yaml.safe_load(open('/home/wyx/projects/SFDFusion/configs/cfg.yaml'))
    cfg = from_dict(config)
    train_dataset = RoadScene_torch(cfg, 'train')
    print(f"成功初始化 PyTorch RoadScene(mode='train') 数据集。将处理 {NUM_TEST_SAMPLES} 个样本。")

    # 关键修改 2: 准备列表来收集每个样本的数据
    ir_list, vi_list, mask_list, name_list = [], [], [], []

    # 2. 循环获取多个样本
    for i in range(NUM_TEST_SAMPLES):
        print(f"正在获取索引为 {i} 的样本...")
        ir_img, vi_img, mask, img_name = train_dataset[i]
        
        # 确保张量有通道维度
        ir_list.append(ir_img.numpy())
        vi_list.append(vi_img.numpy())
        mask_list.append(mask.numpy())
        name_list.append(img_name)

    # 关键修改 3: 将样本列表堆叠成一个批次
    ir_batch = np.stack(ir_list, axis=0)
    vi_batch = np.stack(vi_list, axis=0)
    mask_batch = np.stack(mask_list, axis=0)
    name_batch = np.array(name_list)

    print(f"数据堆叠完成。ir_batch 形状: {ir_batch.shape}")

    # 3. 将批处理结果保存到 .npz 文件
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    np.savez(
        PYTORCH_RESULT_PATH,
        ir_img=ir_batch,
        vi_img=vi_batch,
        mask=mask_batch,
        img_name=name_batch
    )
    print(f"✅ PyTorch 批量输出数据已保存至: {PYTORCH_RESULT_PATH}")
    print("--- PyTorch 环境：任务完成 ---")

if __name__ == '__main__':
    generate_pytorch_batch()

--- PyTorch 环境：开始生成批量待测数据 ---
find mask cache in folder, skip saliency detection
成功初始化 PyTorch RoadScene(mode='train') 数据集。将处理 10 个样本。
正在获取索引为 0 的样本...
正在获取索引为 1 的样本...
正在获取索引为 2 的样本...
正在获取索引为 3 的样本...
正在获取索引为 4 的样本...
正在获取索引为 5 的样本...
正在获取索引为 6 的样本...
正在获取索引为 7 的样本...
正在获取索引为 8 的样本...
正在获取索引为 9 的样本...
数据堆叠完成。ir_batch 形状: (10, 1, 320, 320)
✅ PyTorch 批量输出数据已保存至: /home/wyx/projects/pyTest/dataset_test/output/pytorch_sample.npz
--- PyTorch 环境：任务完成 ---


/home/wyx/miniconda3/envs/SFD/lib/python3.10/site-packages/torchvision/transforms/functional.py:1603: UserWarning: The default value of the antialias parameter of all the resizing transforms (Resize(), RandomResizedCrop(), etc.) will change from None to True in v0.17, in order to be consistent across the PIL and Tensor backends. To suppress this warning, directly pass antialias=True (recommended, future default), antialias=None (current default, which means False for Tensors and True for PIL), or antialias=False (only works on Tensors - PIL will still use antialiasing). This also applies if you are using the inference transforms from the models weights: update the call to weights.transforms(antialias=True).
  warnings.warn(


## models.py

In [10]:
import torch
import numpy as np
from pathlib import Path
import os

# 假设您的 PyTorch 模块代码保存在 SFDFusion_jittor/modules_torch.py
from SFDFusion.modules import Att_Block, Sobelxy, DMRM, Fuse_block, IFFT, AmpFuse, PhaFuse, Fuse, fft

# --- 配置 ---
BATCH_SIZE = 2
CHANNELS = 1
HEIGHT = 64
WIDTH = 64
DMRM_CHANNEL = 8
# 关键修改 1: 定义测试轮次
NUM_TEST_RUNS = 5

# 设置随机种子以保证整个过程可复现
torch.manual_seed(42)
np.random.seed(42)

# --- 输出路径 ---
CWD = Path.cwd()
OUTPUT_DIR = CWD / 'pyTest' / 'modules_test' / 'output'
PYTORCH_RESULT_PATH = OUTPUT_DIR / 'pytorch_modules.npz'

def get_nested_attr_pytorch(obj, attr_path):
    """
    安全地获取 PyTorch 模型中的嵌套属性，支持模块列表索引。
    例如, "att.0.weight" 会被正确解析。
    """
    parts = attr_path.split('.')
    current_obj = obj
    for part in parts:
        if part.isdigit():
            # 处理列表/Sequential 中的索引
            current_obj = current_obj[int(part)]
        else:
            # 处理常规属性
            current_obj = getattr(current_obj, part)
    return current_obj

def test_module(module_name, model, inputs, grad_weight_name, run_idx):
    """通用模块测试函数，增加了 run_idx 用于区分不同轮次"""
    print(f"--- [轮次 {run_idx}] 测试 PyTorch 模块: {module_name} ---")
    
    # 清空之前的梯度
    model.zero_grad()
    
    # 前向传播
    model.eval()
    outputs = model(*inputs)
    
    # 后向传播 (需要模型在 train 模式)
    model.train()
    # 将输出统一为元组，以便求和
    output_tuple = outputs if isinstance(outputs, tuple) else (outputs,)
    # 将所有输出张量的和作为 loss
    loss = sum(torch.sum(o) for o in output_tuple)
    loss.backward()
    
    # 准备保存结果的字典
    results = {}
    key_prefix = f'{module_name}_run{run_idx}'
    
    # 保存输入张量
    for i, inp in enumerate(inputs):
        results[f'{key_prefix}_input_{i}'] = inp.detach().numpy()
        
    # 保存输出张量
    if isinstance(outputs, tuple):
        for i, out in enumerate(outputs):
            results[f'{key_prefix}_output_{i}'] = out.detach().numpy()
    else:
        results[f'{key_prefix}_output'] = outputs.detach().numpy()
        
    # 使用辅助函数安全地获取要检查的权重参数
    weight_param = get_nested_attr_pytorch(model, grad_weight_name)
    
    # 保存该参数的梯度
    if weight_param.grad is not None:
        results[f'{key_prefix}_grad'] = weight_param.grad.detach().numpy()
    else:
        # 如果梯度不存在，也明确记录下来
        print(f"警告: 模块 {module_name} 的权重 {grad_weight_name} 没有梯度。")
        # 使用一个 NumPy 数组作为占位符
        results[f'{key_prefix}_grad'] = np.array([0.0], dtype=np.float32)

    # 保存该参数的权重值
    results[f'{key_prefix}_weight'] = weight_param.detach().numpy()
    
    return results

def test_fft_func(run_idx):
    """测试独立的 fft 函数，增加了 run_idx"""
    print(f"--- [轮次 {run_idx}] 测试 PyTorch 函数: fft ---")
    key_prefix = f'fft_run{run_idx}'
    input_tensor = torch.randn(BATCH_SIZE, CHANNELS, HEIGHT, WIDTH)
    amp, pha = fft(input_tensor)
    return {
        f'{key_prefix}_input': input_tensor.numpy(),
        f'{key_prefix}_amp': amp.detach().numpy(),
        f'{key_prefix}_pha': pha.detach().numpy()
    }

def convert_and_save_weights_as_npz(state_dict, path):
    """
    将 PyTorch 的 state_dict 转换为 NumPy 字典并保存为 .npz 文件。
    """
    numpy_dict = {}
    for key, value in state_dict.items():
        numpy_dict[key] = value.cpu().numpy()
    np.savez(path, **numpy_dict)

def main():
    """主函数，执行所有测试并保存结果"""
    all_results = {}
    print(f"将执行 {NUM_TEST_RUNS} 轮批量测试...")

    # 实例化所有模型一次
    att_model = Att_Block(DMRM_CHANNEL, DMRM_CHANNEL)
    sobel_model = Sobelxy(DMRM_CHANNEL)
    dmrm_model = DMRM(CHANNELS, DMRM_CHANNEL)
    fuse_model = Fuse()

    # 关键修正: 将权重保存为通用的 .npz 格式
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    convert_and_save_weights_as_npz(att_model.state_dict(), OUTPUT_DIR / 'att_model_weights.npz')
    convert_and_save_weights_as_npz(dmrm_model.state_dict(), OUTPUT_DIR / 'dmrm_model_weights.npz')
    convert_and_save_weights_as_npz(fuse_model.state_dict(), OUTPUT_DIR / 'fuse_model_weights.npz')
    print(f"PyTorch 模型权重已保存为 .npz 格式至: {OUTPUT_DIR}")

    # (循环和测试的其余部分代码保持不变)
    for i in range(NUM_TEST_RUNS):
        # --- 独立函数测试 ---
        all_results.update(test_fft_func(i))

        # --- 模块测试 ---
        # 1. Att_Block
        att_input = (torch.randn(BATCH_SIZE, DMRM_CHANNEL, HEIGHT, WIDTH),)
        all_results.update(test_module("Att_Block", att_model, att_input, "att.0.weight", i))

        # 2. Sobelxy (只测试前向)
        sobel_input = torch.randn(BATCH_SIZE, DMRM_CHANNEL, HEIGHT, WIDTH)
        sobel_output = sobel_model(sobel_input)
        all_results[f'Sobelxy_run{i}_input'] = sobel_input.detach().numpy()
        all_results[f'Sobelxy_run{i}_output'] = sobel_output.detach().numpy()
        if i == 0:
            all_results['Sobelxy_weight_x'] = sobel_model.convx.weight.detach().numpy()
            all_results['Sobelxy_weight_y'] = sobel_model.convy.weight.detach().numpy()

        # 3. DMRM
        dmrm_input = (torch.randn(BATCH_SIZE, CHANNELS, HEIGHT, WIDTH), torch.randn(BATCH_SIZE, CHANNELS, HEIGHT, WIDTH))
        all_results.update(test_module("DMRM", dmrm_model, dmrm_input, "ir_embed.0.weight", i))

        # 4. Fuse (顶级模块)
        fuse_input = (torch.randn(BATCH_SIZE, CHANNELS, HEIGHT, WIDTH), torch.randn(BATCH_SIZE, CHANNELS, HEIGHT, WIDTH))
        all_results.update(test_module("Fuse", fuse_model, fuse_input, "dmrm.ir_embed.0.weight", i))

    # --- 保存所有结果 ---
    np.savez(PYTORCH_RESULT_PATH, **all_results)
    print(f"\n🎉 所有 {NUM_TEST_RUNS} 轮 PyTorch 基准数据已成功保存至: {PYTORCH_RESULT_PATH}")

if __name__ == "__main__":
    main()

将执行 5 轮批量测试...
PyTorch 模型权重已保存为 .npz 格式至: /home/wyx/projects/pyTest/modules_test/output
--- [轮次 0] 测试 PyTorch 函数: fft ---
--- [轮次 0] 测试 PyTorch 模块: Att_Block ---
--- [轮次 0] 测试 PyTorch 模块: DMRM ---
--- [轮次 0] 测试 PyTorch 模块: Fuse ---
--- [轮次 1] 测试 PyTorch 函数: fft ---
--- [轮次 1] 测试 PyTorch 模块: Att_Block ---
--- [轮次 1] 测试 PyTorch 模块: DMRM ---
--- [轮次 1] 测试 PyTorch 模块: Fuse ---
--- [轮次 2] 测试 PyTorch 函数: fft ---
--- [轮次 2] 测试 PyTorch 模块: Att_Block ---
--- [轮次 2] 测试 PyTorch 模块: DMRM ---
--- [轮次 2] 测试 PyTorch 模块: Fuse ---
--- [轮次 3] 测试 PyTorch 函数: fft ---
--- [轮次 3] 测试 PyTorch 模块: Att_Block ---
--- [轮次 3] 测试 PyTorch 模块: DMRM ---
--- [轮次 3] 测试 PyTorch 模块: Fuse ---
--- [轮次 4] 测试 PyTorch 函数: fft ---
--- [轮次 4] 测试 PyTorch 模块: Att_Block ---
--- [轮次 4] 测试 PyTorch 模块: DMRM ---
--- [轮次 4] 测试 PyTorch 模块: Fuse ---

🎉 所有 5 轮 PyTorch 基准数据已成功保存至: /home/wyx/projects/pyTest/modules_test/output/pytorch_modules.npz


## train.py

In [1]:
import torch
import numpy as np
import yaml
from pathlib import Path
import os
import sys

# 修正: 使用 os.getcwd() 代替 __file__ 来确保在 notebook 中也能运行
# 这会将当前工作目录添加到 python 路径中
sys.path.append(os.getcwd())

# 导入必要的组件
from SFDFusion.modules import Fuse
from SFDFusion.utils.loss import PixelGradLoss, cal_saliency_loss, cal_fre_loss
from SFDFusion.dataset import RoadScene
from SFDFusion.configs import from_dict
import kornia

def get_gradients(model):
    """将模型参数的梯度提取到一个字典中。"""
    grads = {}
    for name, param in model.named_parameters():
        if param.grad is not None:
            grads[name] = param.grad.cpu().numpy()
    return grads

def main():
    print("--- PyTorch 环境：开始生成训练步骤基准数据 ---")

    # --- 1. 配置和初始化 ---
    CWD = Path.cwd()
    OUTPUT_DIR = CWD / 'pyTest' / 'train_test' / 'output'
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    config = yaml.safe_load(open('SFDFusion/configs/cfg.yaml'))
    cfg = from_dict(config)
    torch.manual_seed(cfg.seed)
    np.random.seed(cfg.seed)
    
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"PyTorch 使用设备: {device}")

    # --- 2. 准备模型、优化器、损失函数 ---
    fuse_net = Fuse().to(device)
    optimizer = torch.optim.Adam(fuse_net.parameters(), lr=cfg.lr_i)
    
    loss_ssim = kornia.losses.SSIMLoss(window_size=11).to(device)
    loss_grad_pixel = PixelGradLoss().to(device)

    # --- 3. 准备数据 ---
    print("正在准备数据批次...")
    train_dataset = RoadScene(cfg, 'train')
    data_ir, data_vi, mask, _ = train_dataset[0]
    data_ir = torch.stack([data_ir, data_ir.clone()], dim=0).to(device)
    data_vi = torch.stack([data_vi, data_vi.clone()], dim=0).to(device)
    mask = torch.stack([mask, mask.clone()], dim=0).to(device)
    print(f"数据准备完成, ir_shape: {data_ir.shape}, vi_shape: {data_vi.shape}")

    # --- 4. 保存初始状态 ---
    initial_weights = {name: param.cpu().numpy() for name, param in fuse_net.state_dict().items()}
    
    # --- 5. 执行一个训练步骤 ---
    fuse_net.train()
    optimizer.zero_grad()

    fus_data, amp, pha = fuse_net(data_ir, data_vi)

    content_loss = loss_grad_pixel(data_vi, data_ir, fus_data)
    ssim_loss_v = loss_ssim(data_vi, fus_data)
    ssim_loss_i = loss_ssim(data_ir, fus_data)
    ssim_loss = ssim_loss_i + ssim_loss_v
    saliency_loss = cal_saliency_loss(fus_data, data_ir, data_vi, mask)
    fre_loss = cal_fre_loss(amp, pha, data_ir, data_vi, mask)
    total_loss = (cfg.coeff_content * content_loss + 
                  cfg.coeff_ssim * ssim_loss + 
                  cfg.coeff_saliency * saliency_loss + 
                  cfg.coeff_fre * fre_loss)

    total_loss.backward()
    
    gradients_before_step = get_gradients(fuse_net)

    optimizer.step()

    # --- 6. 保存所有结果 ---
    results_to_save = {
        'input_ir': data_ir.cpu().numpy(),
        'input_vi': data_vi.cpu().numpy(),
        'input_mask': mask.cpu().numpy(),
        **{'init_weight_' + k: v for k, v in initial_weights.items()},
        'output_fus': fus_data.detach().cpu().numpy(),
        'output_amp': amp.detach().cpu().numpy(),
        'output_pha': pha.detach().cpu().numpy(),
        'loss_content': content_loss.detach().cpu().numpy(),
        'loss_ssim': ssim_loss.detach().cpu().numpy(),
        'loss_saliency': saliency_loss.detach().cpu().numpy(),
        'loss_fre': fre_loss.detach().cpu().numpy(),
        'loss_total': total_loss.detach().cpu().numpy(),
        **{'grad_' + k: v for k, v in gradients_before_step.items()},
        **{'updated_weight_' + name: param.cpu().numpy() for name, param in fuse_net.state_dict().items()}
    }

    output_path = OUTPUT_DIR / 'pytorch_train_step_results.npz'
    np.savez(output_path, **results_to_save)
    
    print(f"✅ PyTorch 训练步骤基准数据已保存至: {output_path}")
    print("--- PyTorch 环境：任务完成 ---")

if __name__ == '__main__':
    main()

--- PyTorch 环境：开始生成训练步骤基准数据 ---
PyTorch 使用设备: cuda
正在准备数据批次...
find mask cache in folder, skip saliency detection
数据准备完成, ir_shape: torch.Size([2, 1, 320, 320]), vi_shape: torch.Size([2, 1, 320, 320])


/home/wyx/miniconda3/envs/SFD/lib/python3.10/site-packages/torchvision/transforms/functional.py:1603: UserWarning: The default value of the antialias parameter of all the resizing transforms (Resize(), RandomResizedCrop(), etc.) will change from None to True in v0.17, in order to be consistent across the PIL and Tensor backends. To suppress this warning, directly pass antialias=True (recommended, future default), antialias=None (current default, which means False for Tensors and True for PIL), or antialias=False (only works on Tensors - PIL will still use antialiasing). This also applies if you are using the inference transforms from the models weights: update the call to weights.transforms(antialias=True).
  warnings.warn(


✅ PyTorch 训练步骤基准数据已保存至: /home/wyx/projects/pyTest/train_test/output/pytorch_train_step_results.npz
--- PyTorch 环境：任务完成 ---


## cc测试

In [1]:
# run_cc_torch.py
import numpy as np
import torch

def cc_torch(img1, img2):
    eps = 1e-7
    N, C, _, _ = img1.shape
    img1 = img1.reshape(N, C, -1)
    img2 = img2.reshape(N, C, -1)
    img1 = img1 - img1.mean(dim=-1, keepdim=True)
    img2 = img2 - img2.mean(dim=-1, keepdim=True)
    num = torch.sum(img1 * img2, dim=-1)
    den = torch.sqrt(torch.sum(img1**2, dim=-1)) * torch.sqrt(torch.sum(img2**2, dim=-1))
    return torch.clamp(num / (den + eps), -1.0, 1.0).mean()

# 统一生成输入
np.random.seed(0)
img1_np = np.random.randn(4, 1, 128, 128).astype(np.float32)
img2_np = np.random.randn(4, 1, 128, 128).astype(np.float32)

img1 = torch.from_numpy(img1_np)
img2 = torch.from_numpy(img2_np)

cc_val = cc_torch(img1, img2).item()
print(f"✅ PyTorch cc: {cc_val:.8f}")


✅ PyTorch cc: -0.00610899


## fft测试

In [1]:
import torch
import numpy as np

def test_fft_torch():
    print("=== PyTorch FFT/IFFT测试 ===\n")
    np.random.seed(42)
    H, W = 64, 64
    test_data = np.random.randn(1, 1, H, W).astype(np.float32)
    
    pt_input = torch.from_numpy(test_data)
    pt_fft = torch.fft.rfftn(pt_input, dim=(-2, -1))
    pt_amp = torch.abs(pt_fft)
    pt_pha = torch.angle(pt_fft)
    
    pt_complex = torch.polar(pt_amp, pt_pha)
    pt_ifft = torch.abs(torch.fft.irfftn(pt_complex, s=(H, W), dim=(-2, -1)))

    print(f"输入均值: {pt_input.mean().item():.6f}")
    print(f"FFT幅度均值: {pt_amp.mean().item():.6f}")
    print(f"IFFT重建均值: {pt_ifft.mean().item():.6f}")
    print(f"重建误差: {torch.abs(pt_input - pt_ifft).max().item():.6e}")

    # 频域损失
    from SFDFusion.utils.loss import cal_fre_loss
    mask = (np.random.rand(1, 1, H, W) > 0.5).astype(np.float32)
    pt_mask = torch.from_numpy(mask)
    pt_loss = cal_fre_loss(pt_amp, pt_pha, pt_input, pt_input, pt_mask)
    print(f"PyTorch频域损失: {pt_loss.item():.6f}")
    print("\n✅ PyTorch测试完成！")

if __name__ == "__main__":
    test_fft_torch()


=== PyTorch FFT/IFFT测试 ===

输入均值: 0.016564
FFT幅度均值: 56.726147
IFFT重建均值: 0.795406
重建误差: 6.482534e+00
PyTorch频域损失: 0.057946

✅ PyTorch测试完成！


In [1]:
# test_irfftn_pytorch.py
import numpy as np
import torch
import os

# 参数
N, C, H, W = 1, 1, 8, 8
W_half = W // 2 + 1

# 随机半谱，模拟 rfftn 输出格式
half_spec_np = np.random.randn(N, C, H, W_half, 2).astype(np.float32)
torch_half_spec = torch.from_numpy(half_spec_np)
torch_half_spec_complex = torch.view_as_complex(torch_half_spec)

# 逆变换
torch_output = torch.fft.irfftn(torch_half_spec_complex, s=(H, W), dim=(-2, -1)).numpy()

# 保存
os.makedirs("output", exist_ok=True)
np.save("output/half_spec.npy", half_spec_np)
np.save("output/pytorch_output.npy", torch_output)
print("✅ PyTorch 结果已保存：output/half_spec.npy 和 output/pytorch_output.npy")


✅ PyTorch 结果已保存：output/half_spec.npy 和 output/pytorch_output.npy


## fuse.py

In [1]:
import torch
import numpy as np
import os
from pathlib import Path
import shutil
from PIL import Image
import argparse

# 导入原始 PyTorch 项目的组件
from SFDFusion.modules import Fuse
from SFDFusion.utils.img_read import img_read, img_save, ycbcr_to_rgb, tensor_to_image

def setup_fuse_test_environment():
    """创建测试所需的所有目录、文件和模型权重。"""
    print("--- 正在设置 fuse.py 测试环境 ---")
    CWD = Path.cwd()
    TEST_DIR = CWD / 'pyTest' / 'fuse_test'
    # （这部分代码和之前一样，为了完整性一并提供）
    INPUT_DIR = TEST_DIR / 'input'
    OUTPUT_DIR = TEST_DIR / 'output'
    IR_DIR = INPUT_DIR / 'ir'
    VI_DIR = INPUT_DIR / 'vi'
    PT_OUT_DIR = OUTPUT_DIR / 'pytorch'
    MODEL_DIR = TEST_DIR / 'models'

    if TEST_DIR.exists():
        shutil.rmtree(TEST_DIR)
    for path in [IR_DIR, VI_DIR, PT_OUT_DIR, MODEL_DIR]:
        path.mkdir(parents=True, exist_ok=True)

    np.random.seed(42)
    img_ir = np.random.randint(0, 256, (128, 128), dtype=np.uint8)
    img_vi = np.random.randint(0, 256, (128, 128, 3), dtype=np.uint8)
    Image.fromarray(img_ir, 'L').save(IR_DIR / 'test_image.png')
    Image.fromarray(img_vi, 'RGB').save(VI_DIR / 'test_image.png')
    print(f"测试图像已创建在: {INPUT_DIR}")

    torch.manual_seed(42)
    dummy_fuse_net = Fuse()
    torch.save({'fuse_net': dummy_fuse_net.state_dict()}, MODEL_DIR / 'dummy_model.pth')
    np.savez(
        MODEL_DIR / 'dummy_model_weights.npz',
        **{k: v.cpu().numpy() for k, v in dummy_fuse_net.state_dict().items()}
    )
    print(f"测试模型已保存在: {MODEL_DIR}")
    return TEST_DIR

def run_pytorch_fuse(test_dir):
    """运行 PyTorch fuse.py 逻辑并保存结果。"""
    print("\n--- 开始执行 PyTorch fuse.py 基准测试 (已修正取整) ---")
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    fuse_net = Fuse()
    ckpt_path = test_dir / 'models' / 'dummy_model.pth'
    ckpt = torch.load(ckpt_path, map_location=device)
    fuse_net.load_state_dict(ckpt['fuse_net'])
    fuse_net.to(device)
    fuse_net.eval()
    
    ir_path = test_dir / 'input' / 'ir'
    vi_path = test_dir / 'input' / 'vi'
    img_name = 'test_image.png'

    ir_img = img_read(ir_path / img_name, mode='L').unsqueeze(0)
    vi_y_img, vi_cbcr_img = img_read(vi_path / img_name, mode='YCbCr')
    vi_y_img = vi_y_img.unsqueeze(0)
    vi_cbcr_img = vi_cbcr_img.unsqueeze(0)

    _, _, h, w = ir_img.shape
    if h % 2 != 0 or w % 2 != 0:
        h, w = h // 2 * 2, w // 2 * 2
        ir_img = ir_img[:, :, :h, :w]
        vi_y_img = vi_y_img[:, :, :h, :w]
        vi_cbcr_img = vi_cbcr_img[:, :, :h, :w]
        
    data_ir = ir_img.to(device)
    data_vi_y = vi_y_img.to(device)

    with torch.no_grad():
        fus_data, _, _ = fuse_net(data_ir, data_vi_y)

    pt_out_dir = test_dir / 'output' / 'pytorch'
    
    # --- 关键修正：添加 np.round() ---
    fi_gray = np.squeeze((fus_data * 255).cpu().numpy())
    fi_gray = np.round(fi_gray).astype(np.uint8) 
    img_save(fi_gray, img_name, pt_out_dir / 'gray')
    print(f"✅ PyTorch [灰度] 融合结果已保存至: {pt_out_dir / 'gray'}")

    data_vi_cbcr = vi_cbcr_img.to(device)
    fi_rgb = torch.cat((fus_data, data_vi_cbcr), dim=1)
    fi_rgb = ycbcr_to_rgb(fi_rgb)
    fi_rgb = tensor_to_image(fi_rgb) * 255
    fi_rgb = np.round(fi_rgb).astype(np.uint8)
    img_save(fi_rgb, img_name, pt_out_dir / 'rgb', mode='RGB')
    print(f"✅ PyTorch [RGB] 融合结果已保存至: {pt_out_dir / 'rgb'}")

test_directory = setup_fuse_test_environment()
run_pytorch_fuse(test_directory)

--- 正在设置 fuse.py 测试环境 ---
测试图像已创建在: /home/wyx/projects/pyTest/fuse_test/input
测试模型已保存在: /home/wyx/projects/pyTest/fuse_test/models

--- 开始执行 PyTorch fuse.py 基准测试 (已修正取整) ---


/tmp/ipykernel_429489/23018048.py:34: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(img_ir, 'L').save(IR_DIR / 'test_image.png')
/tmp/ipykernel_429489/23018048.py:35: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(img_vi, 'RGB').save(VI_DIR / 'test_image.png')


✅ PyTorch [灰度] 融合结果已保存至: /home/wyx/projects/pyTest/fuse_test/output/pytorch/gray
✅ PyTorch [RGB] 融合结果已保存至: /home/wyx/projects/pyTest/fuse_test/output/pytorch/rgb


## 推理

In [3]:
import os
os.chdir('/home/wyx/projects/SFDFusion')
!python3 fuse.py
!python3 val.py

2025-07-17 15:59:41,912 | fuse.py[line:47] | INFO | 成功从 .pth 文件加载权重: models/pytorch_50epoch.pth
2025-07-17 15:59:41,916 | fuse.py[line:67] | INFO | 开始融合 3 张图像...
100%|█████████████████████████████████████████████| 3/3 [00:00<00:00,  3.16it/s]
2025-07-17 15:59:42,867 | fuse.py[line:121] | INFO | 所有图像融合完成！平均处理时间: 0.0036 秒/张。
2025-07-17 15:59:42,868 | fuse.py[line:122] | INFO | 灰度结果保存在: test_result/fuse_result/gray
2025-07-17 15:59:42,868 | fuse.py[line:124] | INFO | RGB 结果保存在: test_result/fuse_result/rgb
2025-07-17 15:59:47,319 | dataset.py[line:142] | INFO | load 44 images
2025-07-17 15:59:47,503 | val.py[line:46] | INFO | fusing images ...
100%|███████████████████████████████████████████| 44/44 [00:03<00:00, 12.02it/s]
2025-07-17 15:59:51,168 | val.py[line:66] | INFO | fusing images done!
2025-07-17 15:59:51,168 | val.py[line:67] | INFO | time: 0.058575s
2025-07-17 15:59:51,168 | dataset.py[line:142] | INFO | load 44 images
2025-07-17 15:59:51,169 | val.py[line:79] | INFO | evaluating 

## 推理测试

In [ ]:
import os
import torch
import numpy as np
import yaml
import sys
from pathlib import Path
import logging
from PIL import Image
import pickle
from collections import OrderedDict

# --- 环境设置 ---
# 假设此脚本在 SFDFusion (PyTorch原版) 项目的根目录下运行
# 并假设 SFDFusion_jittor 在其同级目录
try:
    original_pytorch_project_path = '/home/wyx/projects/SFDFusion'
    if original_pytorch_project_path not in sys.path:
        sys.path.append(original_pytorch_project_path)
    os.chdir(original_pytorch_project_path)
except FileNotFoundError:
    print("错误: 无法切换到PyTorch项目目录。请检查路径。")
    sys.exit(1)

from modules import Fuse
from configs import from_dict

# --- 配置 ---
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CONFIG_PATH = 'configs/cfg.yaml'
WEIGHTS_PATH = '../initial_weights.bin'
IR_IMAGE_DIR = '../SFDFusion_jittor/RoadScene/ir'
VI_IMAGE_DIR = '../SFDFusion_jittor/RoadScene/vi'
OUTPUT_DIR = '../pyTest/forward/pytorch'
NUM_IMAGES_TO_TEST = 4 # 测试的图片数量

def load_images_to_torch_batch(ir_dir, vi_dir, img_size, max_images):
    """加载图片并创建PyTorch批次数据"""
    ir_paths = sorted(list(Path(ir_dir).glob('*.jpg')))[:max_images]
    vi_paths = sorted(list(Path(vi_dir).glob('*.jpg')))[:max_images]
    
    batch_ir_np, batch_vi_np = [], []
    filenames = []

    for ir_path, vi_path in zip(ir_paths, vi_paths):
        ir_img = Image.open(ir_path).convert('L').resize((img_size, img_size), Image.BILINEAR)
        vi_img = Image.open(vi_path).convert('L').resize((img_size, img_size), Image.BILINEAR)
        
        ir_np = np.array(ir_img, dtype=np.float32) / 255.0
        vi_np = np.array(vi_img, dtype=np.float32) / 255.0
        
        batch_ir_np.append(ir_np[np.newaxis, :])
        batch_vi_np.append(vi_np[np.newaxis, :])
        filenames.append(ir_path.name)
        
    data_ir = torch.from_numpy(np.stack(batch_ir_np, axis=0)).to(DEVICE)
    data_vi = torch.from_numpy(np.stack(batch_vi_np, axis=0)).to(DEVICE)
    return data_ir, data_vi, filenames

def main():
    logging.info(f"--- PyTorch 正向传播端到端测试 ---")
    logging.info(f"使用设备: {DEVICE}")

    # 1. 加载配置
    config = yaml.safe_load(open(CONFIG_PATH))
    cfg = from_dict(config)
    img_size = cfg.img_size

    # 2. 初始化模型并加载权重
    fuse_net = Fuse().to(DEVICE)
    try:
        logging.info(f"正在从 {WEIGHTS_PATH} 加载权重...")
        with open(WEIGHTS_PATH, 'rb') as f:
            jittor_weights = pickle.load(f)
        
        pytorch_state_dict = OrderedDict()
        for key, value in jittor_weights.items():
            pytorch_state_dict[key] = torch.from_numpy(value)
        
        fuse_net.load_state_dict(pytorch_state_dict)
        logging.info("✅ 成功加载并转换权重。")

    except Exception as e:
        logging.error(f"❌ 加载权重失败: {e}")
        return
        
    fuse_net.eval() # 设置为评估模式

    # 3. 加载真实图片
    logging.info(f"正在从 {IR_IMAGE_DIR} 和 {VI_IMAGE_DIR} 加载 {NUM_IMAGES_TO_TEST} 张图片...")
    data_ir, data_vi, filenames = load_images_to_torch_batch(IR_IMAGE_DIR, VI_IMAGE_DIR, img_size, NUM_IMAGES_TO_TEST)
    logging.info(f"输入数据尺寸: {data_ir.shape}")

    # 4. 执行正向传播
    with torch.no_grad(): # 关闭梯度计算
        fus_data, _, _ = fuse_net(data_ir, data_vi)
    
    fus_data_np = fus_data.cpu().numpy()
    logging.info(f"输出数据尺寸: {fus_data_np.shape}")

    # 5. 保存融合结果
    Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
    for i, filename in enumerate(filenames):
        fused_image_np = fus_data_np[i, 0, :, :]
        fused_image_np = np.clip(fused_image_np * 255, 0, 255).astype(np.uint8)
        img = Image.fromarray(fused_image_np)
        save_path = Path(OUTPUT_DIR) / filename
        img.save(save_path)
        logging.info(f"已保存融合图片到: {save_path}")

    logging.info("✅ PyTorch 正向传播测试完成。")

if __name__ == '__main__':
    main()

--- PyTorch 正向传播端到端测试 ---
使用设备: cuda
正在从 ../initial_weights.bin 加载权重...
✅ 成功加载并转换权重。
正在从 ../SFDFusion_jittor/RoadScene/ir 和 ../SFDFusion_jittor/RoadScene/vi 加载 4 张图片...
输入数据尺寸: torch.Size([4, 1, 320, 320])
输出数据尺寸: (4, 1, 320, 320)
已保存融合图片到: ../pyTest/forward/pytorch/FLIR_00006.jpg
已保存融合图片到: ../pyTest/forward/pytorch/FLIR_00018.jpg
已保存融合图片到: ../pyTest/forward/pytorch/FLIR_00060.jpg
已保存融合图片到: ../pyTest/forward/pytorch/FLIR_00122.jpg
✅ PyTorch 正向传播测试完成。


## 训练

In [3]:
!cd SFDFusion&&python3 train.py

wandb: Currently logged in as: wangyuxie (wangyuxie-njust). Use `wandb login --relogin` to force relogin
wandb: Appending key for api.wandb.ai to your netrc file: /home/wyx/.netrc
wandb: wandb version 0.21.0 is available!  To upgrade, please run:
wandb:  $ pip install wandb --upgrade
wandb: Tracking run with wandb version 0.17.4
wandb: Run data is saved locally in /home/wyx/projects/SFDFusion/wandb/run-20250715_142451-lkndijsz
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run RoadScene_pytorch_50epoch
wandb: ⭐️ View project at https://wandb.ai/wangyuxie-njust/SFDFusion_compare
wandb: 🚀 View run at https://wandb.ai/wangyuxie-njust/SFDFusion_compare/runs/lkndijsz
2025-07-15 14:24:57,696 | dataset.py[line:142] | INFO | load 177 images
find mask cache in folder, skip saliency detection
2025-07-15 14:24:57,700 | dataset.py[line:24] | INFO | find mask cache in folder, skip saliency detection
2025-07-15 14:24:57,701 | train.py[line:101] | INFO | Start training...
  0%|       

## 使用相同权重训练

In [2]:
import os
os.chdir('/home/wyx/projects/SFDFusion')
!python3 train.py --load_initial_weights ../initial_weights.bin

2025-07-17 15:48:04,454 | train.py[line:71] | INFO | Loading initial weights from: ../initial_weights.bin
2025-07-17 15:48:04,470 | train.py[line:80] | INFO | ✅ Successfully loaded initial weights.
wandb: Currently logged in as: wangyuxie (wangyuxie-njust). Use `wandb login --relogin` to force relogin
wandb: Appending key for api.wandb.ai to your netrc file: /home/wyx/.netrc
wandb: wandb version 0.21.0 is available!  To upgrade, please run:
wandb:  $ pip install wandb --upgrade
wandb: Tracking run with wandb version 0.17.4
wandb: Run data is saved locally in /home/wyx/projects/SFDFusion/wandb/run-20250717_154813-ii55cn4e
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run RoadScene_pytorch_50epoch
wandb: ⭐️ View project at https://wandb.ai/wangyuxie-njust/SFDFusion_compare
wandb: 🚀 View run at https://wandb.ai/wangyuxie-njust/SFDFusion_compare/runs/ii55cn4e
2025-07-17 15:48:18,708 | dataset.py[line:142] | INFO | load 177 images
find mask cache in folder, skip saliency de

## 输出初始权重

In [1]:
!cd SFDFusion&&python3 train.py --save_jittor_weights ../initial_weights.bin

2025-07-15 12:23:32,515 | train.py[line:61] | INFO | Saving initial model weights for Jittor to: ../initial_weights.bin
2025-07-15 12:23:32,525 | train.py[line:65] | INFO | ✅ Initial weights saved. Exiting.
nohup: ignoring input and appending output to 'nohup.out'


## loss 重对比

In [5]:
import os
import torch
import numpy as np
import yaml
import sys
from pathlib import Path
import logging
from PIL import Image
import kornia

# --- 设置环境 ---
os.chdir('/home/wyx/projects/SFDFusion')
project_root = Path().resolve()
sys.path.append(str(project_root.parent))

from SFDFusion.modules import Fuse
from SFDFusion.utils.loss import PixelGradLoss, cal_saliency_loss, cal_fre_loss
from SFDFusion.configs import from_dict

# 关闭不必要的日志输出
logging.basicConfig(level=logging.INFO, format='%(message)s')

def load_images_to_batch(ir_dir, vi_dir, img_size, device, max_images=None):
    """
    从指定目录加载多张 JPG 图片，并将其组织成批次数据。
    通过 max_images 参数限制加载的图片数量。
    """
    ir_images = sorted(list(Path(ir_dir).glob('*.jpg')))
    vi_images = sorted(list(Path(vi_dir).glob('*.jpg')))

    assert len(ir_images) == len(vi_images), "IR和VI目录中的图片数量不一致。"
    assert all(ir_img.name == vi_img.name for ir_img, vi_img in zip(ir_images, vi_images)), \
        "IR和VI目录中的图片文件名不匹配。"

    if max_images is not None:
        ir_images = ir_images[:max_images]
        vi_images = vi_images[:max_images]

    batch_ir_tensors = []
    batch_vi_tensors = []
    batch_mask_tensors = []

    print(f"正在加载 {len(ir_images)} 对图片 (限制数量)。")

    for ir_img_path, vi_img_path in zip(ir_images, vi_images):
        try:
            ir_img = Image.open(ir_img_path).convert('L')
            vi_img = Image.open(vi_img_path).convert('L')

            ir_img = ir_img.resize((img_size, img_size), Image.BILINEAR)
            vi_img = vi_img.resize((img_size, img_size), Image.BILINEAR)

            ir_np = np.array(ir_img).astype(np.float32) / 255.0
            vi_np = np.array(vi_img).astype(np.float32) / 255.0

            batch_ir_tensors.append(torch.from_numpy(np.expand_dims(ir_np, axis=0)).to(device))
            batch_vi_tensors.append(torch.from_numpy(np.expand_dims(vi_np, axis=0)).to(device))
            batch_mask_tensors.append(torch.ones_like(torch.from_numpy(np.expand_dims(ir_np, axis=0))).to(device))

        except Exception as e:
            print(f"加载图片失败: {ir_img_path.name} 或 {vi_img_path.name}. 错误: {e}")
            continue

    if not batch_ir_tensors:
        raise ValueError("未能加载任何图片。请检查路径和图片格式。")

    data_ir = torch.stack(batch_ir_tensors, dim=0)
    data_vi = torch.stack(batch_vi_tensors, dim=0)
    mask = torch.stack(batch_mask_tensors, dim=0)

    return data_ir, data_vi, mask

def run_loss_calculation(fuse_net, loss_ssim, loss_grad_pixel, data_ir, data_vi, mask, description="", device='cpu'):
    """
    Helper function to run forward pass and calculate losses for given data.
    """
    # 确保输入数据在正确设备上
    data_ir = data_ir.to(device)
    data_vi = data_vi.to(device)
    mask = mask.to(device)

    with torch.no_grad():
        fus_data, amp, pha = fuse_net(data_ir, data_vi)

        content_loss = loss_grad_pixel(data_vi, data_ir, fus_data)
        ssim_loss_v = loss_ssim(data_vi, fus_data)
        ssim_loss_i = loss_ssim(data_ir, fus_data)
        ssim_loss = ssim_loss_i + ssim_loss_v
        saliency_loss = cal_saliency_loss(fus_data, data_ir, data_vi, mask)
        fre_loss = cal_fre_loss(amp, pha, data_ir, data_vi, mask)

    print(f"\n--- 计算得到的损失值 (PyTorch) - {description} ---")
    print(f"{'Content Loss:':<20} {content_loss.item():.8f}")
    print(f"{'SSIM Loss:':<20} {ssim_loss.item():.8f}")
    print(f"{'Saliency Loss:':<20} {saliency_loss.item():.8f}")
    print(f"{'Frequency Loss:':<20} {fre_loss.item():.8f}")
    print("-" * 40)


def calculate_losses_pytorch():
    """
    使用图片、全零和全一输入计算并打印 PyTorch 版本中各个损失函数的值。
    """
    print("\n--- PyTorch Loss Calculation ---")

    # --- 1. 配置和初始化 ---
    try:
        config = yaml.safe_load(open('configs/cfg.yaml'))
        cfg = from_dict(config)
    except FileNotFoundError:
        print("错误: 无法找到 configs/cfg.yaml。请确保从 SFDFusion 目录下运行此脚本。")
        return

    seed = 42
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"使用设备: {device}")
    
    # --- 准备模型和损失函数 ---
    fuse_net = Fuse().to(device)

    weight_file = '/home/wyx/projects/initial_weights.bin'
    try:
        import pickle
        print(f"正在从 {weight_file} 加载权重...")
        with open(weight_file, 'rb') as f:
            weights_data_np = pickle.load(f)
        
        pytorch_state_dict = {k: torch.from_numpy(v).to(device) for k, v in weights_data_np.items()}
        
        fuse_net.load_state_dict(pytorch_state_dict)
        print("✅ 成功加载初始权重。")
    except Exception as e:
        print(f"❌ 加载权重失败: {e}")
        raise

    fuse_net.eval()

    loss_ssim = kornia.losses.SSIMLoss(window_size=11).to(device)
    loss_grad_pixel = PixelGradLoss().to(device)

    # --- 2. 加载图片数据并计算损失 ---
    ir_image_dir = '/home/wyx/projects/SFDFusion_jittor/RoadScene/ir'
    vi_image_dir = '/home/wyx/projects/SFDFusion_jittor/RoadScene/vi'
    img_size = cfg.img_size

    num_images_to_load = 4 
    try:
        data_ir_real, data_vi_real, mask_real = load_images_to_batch(ir_image_dir, vi_image_dir, img_size, device, max_images=num_images_to_load)
        print(f"已加载 {data_ir_real.shape[0]} 对图片，尺寸: (B, C, H, W) = ({data_ir_real.shape[0]}, 1, {img_size}, {img_size})")
        run_loss_calculation(fuse_net, loss_ssim, loss_grad_pixel, data_ir_real, data_vi_real, mask_real, "真实图片输入", device)
    except ValueError as e:
        print(f"真实数据加载错误: {e}")

    # --- 3. 全零输入测试 ---
    print("\n--- PyTorch 全零输入测试 ---")
    # batch_size 使用图片加载的实际 batch_size，以保持一致性
    batch_size_test = data_ir_real.shape[0] if 'data_ir_real' in locals() else cfg.batch_size 
    dummy_input_shape = (batch_size_test, 1, img_size, img_size)

    data_ir_zeros = torch.zeros(dummy_input_shape, device=device)
    data_vi_zeros = torch.zeros(dummy_input_shape, device=device)
    mask_zeros = torch.zeros(dummy_input_shape, device=device) # 掩码也设为全零，或根据需要全一

    run_loss_calculation(fuse_net, loss_ssim, loss_grad_pixel, data_ir_zeros, data_vi_zeros, mask_zeros, "全零输入", device)

    # --- 4. 全一输入测试 ---
    print("\n--- PyTorch 全一输入测试 ---")
    data_ir_ones = torch.ones(dummy_input_shape, device=device)
    data_vi_ones = torch.ones(dummy_input_shape, device=device)
    mask_ones = torch.ones(dummy_input_shape, device=device)
    
    run_loss_calculation(fuse_net, loss_ssim, loss_grad_pixel, data_ir_ones, data_vi_ones, mask_ones, "全一输入", device)


if __name__ == '__main__':
    calculate_losses_pytorch()


--- PyTorch Loss Calculation ---
使用设备: cuda
正在从 /home/wyx/projects/initial_weights.bin 加载权重...
✅ 成功加载初始权重。
正在加载 4 对图片 (限制数量)。
已加载 4 对图片，尺寸: (B, C, H, W) = (4, 1, 320, 320)

--- 计算得到的损失值 (PyTorch) - 真实图片输入 ---
Content Loss:        3.39312840
SSIM Loss:           0.55559087
Saliency Loss:       1.16442549
Frequency Loss:      -0.03191719
----------------------------------------

--- PyTorch 全零输入测试 ---

--- 计算得到的损失值 (PyTorch) - 全零输入 ---
Content Loss:        3.71869898
SSIM Loss:           0.99975938
Saliency Loss:       0.61625862
Frequency Loss:      0.00000000
----------------------------------------

--- PyTorch 全一输入测试 ---

--- 计算得到的损失值 (PyTorch) - 全一输入 ---
Content Loss:        3.25743580
SSIM Loss:           0.29795757
Saliency Loss:       2.67212129
Frequency Loss:      0.00000000
----------------------------------------


## 反向传播

In [1]:
import os
import torch
import numpy as np
import yaml
import sys
from pathlib import Path
import logging
from PIL import Image # 导入 PIL 库
import kornia

# --- 设置环境 ---
try:
    os.chdir('/home/wyx/projects/SFDFusion')
    project_root = Path().resolve()
    if str(project_root.parent) not in sys.path:
        sys.path.append(str(project_root.parent))
except FileNotFoundError:
    print("错误: 无法切换到项目目录。请检查路径或手动设置当前工作目录。")
    sys.exit(1)


from SFDFusion.modules import Fuse
from SFDFusion.utils.loss import PixelGradLoss, cal_saliency_loss, cal_fre_loss
from SFDFusion.configs import from_dict

# 关闭不必要的日志输出
logging.basicConfig(level=logging.INFO, format='%(message)s')

def load_images_to_pytorch_batch(ir_dir, vi_dir, img_size, device, max_images=None):
    """
    从指定目录加载多张 JPG 图片，并将其组织成 PyTorch 批次数据。
    返回的 Tensor 会设置 requires_grad=True。
    """
    ir_images = sorted(list(Path(ir_dir).glob('*.jpg')))
    vi_images = sorted(list(Path(vi_dir).glob('*.jpg')))

    assert len(ir_images) == len(vi_images), "IR和VI目录中的图片数量不一致。"
    assert all(ir_img.name == vi_img.name for ir_img, vi_img in zip(ir_images, vi_images)), \
        "IR和VI目录中的图片文件名不匹配。"

    if max_images is not None:
        ir_images = ir_images[:max_images]
        vi_images = vi_images[:max_images]

    batch_ir_tensors = []
    batch_vi_tensors = []
    batch_mask_tensors = []

    print(f"正在加载 {len(ir_images)} 对图片进行测试...")

    for ir_img_path, vi_img_path in zip(ir_images, vi_images):
        try:
            ir_img = Image.open(ir_img_path).convert('L')
            vi_img = Image.open(vi_img_path).convert('L')

            ir_img = ir_img.resize((img_size, img_size), Image.BILINEAR)
            vi_img = vi_img.resize((img_size, img_size), Image.BILINEAR)

            ir_np = np.array(ir_img).astype(np.float32) / 255.0
            vi_np = np.array(vi_img).astype(np.float32) / 255.0

            # 确保输入数据需要梯度
            batch_ir_tensors.append(torch.from_numpy(np.expand_dims(ir_np, axis=0)).to(device).requires_grad_(True))
            batch_vi_tensors.append(torch.from_numpy(np.expand_dims(vi_np, axis=0)).to(device).requires_grad_(True))
            batch_mask_tensors.append(torch.ones_like(torch.from_numpy(np.expand_dims(ir_np, axis=0))).to(device)) # mask 通常不需要梯度

        except Exception as e:
            print(f"加载图片失败: {ir_img_path.name} 或 {vi_img_path.name}. 错误: {e}")
            continue

    if not batch_ir_tensors:
        raise ValueError("未能加载任何图片。请检查路径和图片格式。")

    data_ir = torch.stack(batch_ir_tensors, dim=0)
    data_vi = torch.stack(batch_vi_tensors, dim=0)
    mask = torch.stack(batch_mask_tensors, dim=0)

    return data_ir, data_vi, mask


def test_pytorch_backward_with_real_images():
    print("\n--- PyTorch 反向传播测试 (使用真实图片) ---")

    # --- 1. 配置和初始化 ---
    try:
        config = yaml.safe_load(open('configs/cfg.yaml'))
        cfg = from_dict(config)
    except FileNotFoundError:
        print("错误: 无法找到 configs/cfg.yaml。请确保此脚本在 SFDFusion 目录下运行。")
        return

    seed = 42
    np.random.seed(seed)
    torch.manual_seed(seed) # PyTorch 的随机种子
    
    load_initial_weights = '/home/wyx/projects/initial_weights.bin'
    ir_image_dir = '/home/wyx/projects/SFDFusion/RoadScene/ir' # PyTorch 项目的图片路径
    vi_image_dir = '/home/wyx/projects/SFDFusion/RoadScene/vi'
    img_size = cfg.img_size
    num_images_to_load = 4 # 加载的图片数量，保持小规模

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"使用设备: {device}")
    
    # --- 2. 准备模型和损失函数 ---
    fuse_net = Fuse().to(device)

    try:
        import pickle
        print(f"正在从 {load_initial_weights} 加载权重...")
        with open(load_initial_weights, 'rb') as f:
            weights_data_np = pickle.load(f)
        
        pytorch_state_dict = {k: torch.from_numpy(v).to(device) for k, v in weights_data_np.items()}
        
        fuse_net.load_state_dict(pytorch_state_dict)
        print("✅ 成功加载初始权重。")
    except Exception as e:
        print(f"❌ 加载权重失败: {e}")
        raise

    fuse_net.train() 

    loss_ssim = kornia.losses.SSIMLoss(window_size=11).to(device)
    loss_grad_pixel = PixelGradLoss().to(device)

    # --- 3. 加载真实图片数据 ---
    try:
        data_ir, data_vi, mask = load_images_to_pytorch_batch(ir_image_dir, vi_image_dir, img_size, device, num_images_to_load)
    except ValueError as e:
        print(f"加载图片数据失败: {e}")
        return

    print(f"输入数据尺寸: (B, C, H, W) = {data_ir.shape}")

    # --- 4. 前向传播和计算损失 ---
    fus_data, amp, pha = fuse_net(data_ir, data_vi)

    content_loss = loss_grad_pixel(data_vi, data_ir, fus_data)
    ssim_loss_v = loss_ssim(data_vi, fus_data)
    ssim_loss_i = loss_ssim(data_ir, fus_data)
    ssim_loss = ssim_loss_i + ssim_loss_v
    saliency_loss = cal_saliency_loss(fus_data, data_ir, data_vi, mask)
    fre_loss = cal_fre_loss(amp, pha, data_ir, data_vi, mask)

    total_loss = content_loss + ssim_loss + saliency_loss + fre_loss
    
    print("\n--- 计算得到的损失值 (PyTorch) ---")
    print(f"{'Content Loss:':<20} {content_loss.item():.8f}")
    print(f"{'SSIM Loss:':<20} {ssim_loss.item():.8f}")
    print(f"{'Saliency Loss:':<20} {saliency_loss.item():.8f}")
    print(f"{'Frequency Loss:':<20} {fre_loss.item():.8f}")
    print(f"{'Total Loss:':<20} {total_loss.item():.8f}")
    print("-" * 40)

    # --- 5. 反向传播 ---
    print("\n--- 执行 PyTorch 反向传播 ---")
    
    fuse_net.zero_grad() 
    total_loss.backward()
    
    # --- 6. 检查梯度 ---
    print("\n检查模型参数梯度：")
    gradients_found = False
    for name, param in fuse_net.named_parameters():
        if param.requires_grad: 
            if param.grad is not None:
                gradients_found = True
                grad_mean = param.grad.mean().item()
                grad_std = param.grad.std().item()
                print(f"  {name:<30}: Grad Shape={param.grad.shape}, Mean={grad_mean:.6f}, Std={grad_std:.6f}")
            else:
                print(f"  {name:<30}: Grad is None (requires_grad is True, but no grad) - Possible issue!")
            
    if not gradients_found:
        print("警告: 未发现任何参数梯度！请检查模型是否包含可训练参数，以及损失是否与参数有计算图连接。")
    else:
        print("\n✅ PyTorch 反向传播成功，并检测到参数梯度。")
    
    print("-" * 40)

if __name__ == '__main__':
    test_pytorch_backward_with_real_images()


--- PyTorch 反向传播测试 (使用真实图片) ---
使用设备: cuda
正在从 /home/wyx/projects/initial_weights.bin 加载权重...
✅ 成功加载初始权重。
正在加载 4 对图片进行测试...
输入数据尺寸: (B, C, H, W) = torch.Size([4, 1, 320, 320])

--- 计算得到的损失值 (PyTorch) ---
Content Loss:        3.39312840
SSIM Loss:           0.55559087
Saliency Loss:       1.16442549
Frequency Loss:      -0.03191720
Total Loss:          5.08122778
----------------------------------------

--- 执行 PyTorch 反向传播 ---

检查模型参数梯度：
  dmrm.ir_embed.0.weight        : Grad Shape=torch.Size([8, 1, 3, 3]), Mean=0.076574, Std=0.313876
  dmrm.ir_embed.0.bias          : Grad Shape=torch.Size([8]), Mean=0.115452, Std=0.311604
  dmrm.vi_embed.0.weight        : Grad Shape=torch.Size([8, 1, 3, 3]), Mean=-0.016302, Std=0.202283
  dmrm.vi_embed.0.bias          : Grad Shape=torch.Size([8]), Mean=0.046513, Std=0.186596
  dmrm.ir_att1.att.0.weight     : Grad Shape=torch.Size([8, 8, 3, 3]), Mean=0.001049, Std=0.005766
  dmrm.ir_att1.att.0.bias       : Grad Shape=torch.Size([8]), Mean=0.003950, St